# Train one comparison model on the paper's split

Each run trains **one** model with the v4 training code: the same 5,323 training tiles, the same
280 selection tiles for choosing the best epoch, and the same 623 test tiles left untouched.

| `EXPERIMENT` | What it trains | Recipe |
|---|---|---|
| `unet` | U-Net (31.04M), random initialization | full (160 epochs) |
| `dlinknet34_scratch` | D-LinkNet34 (31.10M), random initialization | full |
| `dlinknet34` | D-LinkNet34, ImageNet ResNet-34 encoder (supplementary) | full |
| `deeplabv3_mbv3` | DeepLabV3 MobileNetV3-Large (11.02M), random initialization | full |
| `lraspp_mbv3` | LR-ASPP MobileNetV3-Large (3.22M), random initialization | full |
| `abl_full` | Our network, unchanged (reference row of the ablation) | short (60 epochs) |
| `abl_no_cldice` | Our network without clDice | short |
| `abl_no_gates` | Our network without attention gates | short |
| `abl_no_strip_shift` | Our network with plain 3x3 convolutions and no channel shift | short |
| `abl_no_transformer` | Our network without transformer layers | short |

**How to run**
1. Kaggle, New Notebook, add the dataset `balraj98/deepglobe-road-extraction-dataset`.
2. File, Import Notebook, choose this file. Settings: GPU T4, Internet on.
3. Set `EXPERIMENT` below. For the first run of a new model, set `SMOKE_TEST = True` and run it once
   (a few minutes); if it ends with `DONE`, set it back to `False`.
4. Save Version, Save & Run All. Make one Kaggle notebook per experiment (name it after `EXPERIMENT`).
5. If it ends with `RESUME NEEDED` (Kaggle's 12 hour limit): Add Input, Your Work, this notebook's
   output, and run a new version. It continues from the last epoch.
6. Download `<EXPERIMENT>/best_<EXPERIMENT>.pth`. After 20 training steps the log prints the
   expected time per epoch and in total.

In [ ]:
# Settings
# Options: "unet", "dlinknet34_scratch", "deeplabv3_mbv3", "lraspp_mbv3", "dlinknet34"
EXPERIMENT = "unet"
SMOKE_TEST = False       # True: 2 short epochs on a few tiles, to check that everything runs
TIME_LIMIT_H = 11.0      # stop and save before Kaggle's 12 hour limit
NUM_WORKERS = 4

In [ ]:
# Code from the repository
import os, sys, glob, json, subprocess, shutil, zipfile
REPO_URL = "https://github.com/Dilraj07/Satellite-Imagery-based-Road-Network-Extraction.git"
REPO_REF = "main"          # a branch, or a commit hash to pin the exact code
REPO = "/kaggle/working/repo"
url = REPO_URL
try:
    from kaggle_secrets import UserSecretsClient
    url = REPO_URL.replace("https://", "https://" + UserSecretsClient().get_secret("GITHUB_TOKEN") + "@")
except Exception:
    pass
env = {**os.environ, "GIT_TERMINAL_PROMPT": "0"}
if not os.path.isdir(REPO):
    if subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", REPO_REF, url, REPO], env=env).returncode:
        if subprocess.run(["git", "clone", "-q", url, REPO], env=env).returncode:        # REPO_REF is a commit
            raise RuntimeError("Could not download the repository: make it public, or add the Kaggle secret GITHUB_TOKEN.")
        subprocess.run(["git", "-C", REPO, "checkout", "-q", REPO_REF], check=True)
else:
    if not subprocess.run(["git", "-C", REPO, "fetch", "-q", "--depth", "1", url, REPO_REF], env=env).returncode:
        subprocess.run(["git", "-C", REPO, "checkout", "-q", "FETCH_HEAD"], env=env)
subprocess.run(["git", "-C", REPO, "remote", "set-url", "origin", REPO_URL], check=True)   # no token in the saved output
V4 =os.path.join(REPO, "models", "best_model_v4.pth")   # the paper's model, included in the repository
print(subprocess.run(["git", "-C", REPO, "log", "-1", "--format=%h %s"], capture_output=True, text=True).stdout)
try:
    import skimage
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "scikit-image"], check=True)
sys.path.insert(0, os.path.join(REPO, "scripts", "experiments"))
import numpy as np, pandas as pd, torch
import exp_data as D
import exp_torch as T
DEVICE = torch.device("cuda")
print("torch", torch.__version__, "|", torch.cuda.get_device_name(0))

In [ ]:
# Split (identical to train_v4; checks the 623 test tiles against the paper)
spec = T.EXPERIMENTS[EXPERIMENT]
assert spec.get("dataset", "deepglobe") == "deepglobe", "use notebooks/massachusetts.ipynb for this experiment"
DG = D.find_deepglobe()
split = D.deepglobe_split(DG)
train_items, select_items = D.deepglobe_items(DG, split["train"]), D.deepglobe_items(DG, split["select"])
print({k: len(v) for k, v in split.items()}, "| test sha1", D.sha1(split["test"]))
OUT = f"/kaggle/working/{EXPERIMENT}" + ("_smoke" if SMOKE_TEST else "")   # a smoke run is never resumed
os.makedirs(OUT, exist_ok=True)
json.dump({k: D.sha1(v) for k, v in split.items()}, open(f"{OUT}/split_sha1.json", "w"), indent=1)

In [ ]:
# Train
status = T.train_experiment(EXPERIMENT, train_items, select_items, OUT, test_sha1=D.TEST_SHA1,
                            time_limit_h=TIME_LIMIT_H, num_workers=NUM_WORKERS, smoke_test=SMOKE_TEST)

In [ ]:
# Curves
import matplotlib.pyplot as plt
h = pd.read_csv(f"{OUT}/history.csv")
fig, a = plt.subplots(1, 2, figsize=(10, 3.5))
a[0].plot(h.epoch, h.train_loss); a[0].set_xlabel("epoch"); a[0].set_ylabel("training loss")
a[1].plot(h.epoch, 100 * h.val_iou, label="model"); a[1].plot(h.epoch, 100 * h.ema_iou, label="EMA")
a[1].set_xlabel("epoch"); a[1].set_ylabel("selection IoU (%)"); a[1].legend()
fig.tight_layout(); fig.savefig(f"{OUT}/training_curves.png", dpi=150)
print(status, sorted(os.listdir(OUT)))